# GoPax 5 min data OHCLV

In [1]:
# !pip install ccxt

In [6]:
import pandas as pd
import os
import datetime as dt
import numpy as np
#!/usr/bin/python
import ccxt
import calendar
from datetime import datetime, date, timedelta

df = pd.DataFrame(index=pd.date_range(start = dt.datetime(2020,1,1), end = dt.datetime.now(), freq='M'))
month_list=df.index.to_series().apply(lambda x: dt.datetime.strftime(x, '%Y-%m'))

In [7]:
print(month_list)

2020-01-31    2020-01
2020-02-29    2020-02
2020-03-31    2020-03
2020-04-30    2020-04
2020-05-31    2020-05
2020-06-30    2020-06
2020-07-31    2020-07
2020-08-31    2020-08
2020-09-30    2020-09
2020-10-31    2020-10
2020-11-30    2020-11
2020-12-31    2020-12
2021-01-31    2021-01
2021-02-28    2021-02
2021-03-31    2021-03
2021-04-30    2021-04
Freq: M, dtype: object


In [36]:
month_list.tolist()

['2020-01',
 '2020-02',
 '2020-03',
 '2020-04',
 '2020-05',
 '2020-06',
 '2020-07',
 '2020-08',
 '2020-09',
 '2020-10',
 '2020-11',
 '2020-12',
 '2021-01',
 '2021-02',
 '2021-03',
 '2021-04']

In [39]:
year = 2020
month = 1
num_days = calendar.monthrange(year, month)[1]
days = [datetime.date(year, month, day) for day in range(1, num_days+1)]

TypeError: descriptor 'date' for 'datetime.datetime' objects doesn't apply to a 'int' object

In [87]:
from datetime import date, timedelta, datetime

def days_cur_month(input_month_list):
    
    res=[]
    for input_month in input_month_list:
        year=int(input_month[0:4])
        month=input_month[-2:]
        month=int(month)
        
        print(input_month)
        m = month
        print(month)
        y=year
#         y = datetime.now().year
        if m<12:
            ndays = (date(y, m+1, 1) - date(y, m, 1)).days
        else:
            ndays=(date(y+1, 1, 1) - date(y, m, 1)).days
        d1 = date(y, m, 1)
        d2 = date(y, m, ndays)
        delta = d2 - d1
        res.append( {input_month:[   (d1 + timedelta(days=i)).strftime('%Y-%m-%d') for i in range(delta.days + 1)] })


        
    return res

#days_cur_month()#

In [88]:
days_cur_month(month_list.tolist())

2020-01
1
2020-02
2
2020-03
3
2020-04
4
2020-05
5
2020-06
6
2020-07
7
2020-08
8
2020-09
9
2020-10
10
2020-11
11
2020-12
12
2021-01
1
2021-02
2
2021-03
3
2021-04
4


[{'2020-01': ['2020-01-01',
   '2020-01-02',
   '2020-01-03',
   '2020-01-04',
   '2020-01-05',
   '2020-01-06',
   '2020-01-07',
   '2020-01-08',
   '2020-01-09',
   '2020-01-10',
   '2020-01-11',
   '2020-01-12',
   '2020-01-13',
   '2020-01-14',
   '2020-01-15',
   '2020-01-16',
   '2020-01-17',
   '2020-01-18',
   '2020-01-19',
   '2020-01-20',
   '2020-01-21',
   '2020-01-22',
   '2020-01-23',
   '2020-01-24',
   '2020-01-25',
   '2020-01-26',
   '2020-01-27',
   '2020-01-28',
   '2020-01-29',
   '2020-01-30',
   '2020-01-31']},
 {'2020-02': ['2020-02-01',
   '2020-02-02',
   '2020-02-03',
   '2020-02-04',
   '2020-02-05',
   '2020-02-06',
   '2020-02-07',
   '2020-02-08',
   '2020-02-09',
   '2020-02-10',
   '2020-02-11',
   '2020-02-12',
   '2020-02-13',
   '2020-02-14',
   '2020-02-15',
   '2020-02-16',
   '2020-02-17',
   '2020-02-18',
   '2020-02-19',
   '2020-02-20',
   '2020-02-21',
   '2020-02-22',
   '2020-02-23',
   '2020-02-24',
   '2020-02-25',
   '2020-02-26',
   '202

In [ ]:
coin_list=[    
]

In [21]:
binance = ccxt.binance()

def min_ohlcv(dt, pair, limit):
    # UTC native object
    since = calendar.timegm(dt.utctimetuple())*1000
    ohlcv1 = binance.fetch_ohlcv(symbol=pair, timeframe='1m', since=since, limit=limit)
    ohlcv2 = binance.fetch_ohlcv(symbol=pair, timeframe='1m', since=since, limit=limit)
    ohlcv = ohlcv1 + ohlcv2
    return ohlcv

def ohlcv(dt, pair, period='1d'):
    ohlcv = []
    limit = 1000
    if period == '1m':
        limit = 720
    elif period == '1d':
        limit = 365
    elif period == '1h':
        limit = 24
    elif period == '5m':
        limit = 288
    for i in dt:
        start_dt = datetime.strptime(i, "%Y%m%d")
        since = calendar.timegm(start_dt.utctimetuple())*1000
        if period == '1m':
            ohlcv.extend(min_ohlcv(start_dt, pair, limit))
        else:
            ohlcv.extend(binance.fetch_ohlcv(symbol=pair, timeframe=period, since=since, limit=limit))
    df = pd.DataFrame(ohlcv, columns = ['Time', 'Open', 'High', 'Low', 'Close', 'Volume'])
    df['Time'] = [datetime.fromtimestamp(float(time)/1000) for time in df['Time']]
    df['Open'] = df['Open'].astype(np.float64)
    df['High'] = df['High'].astype(np.float64)
    df['Low'] = df['Low'].astype(np.float64)
    df['Close'] = df['Close'].astype(np.float64)
    df['Volume'] = df['Volume'].astype(np.float64)
    df.set_index('Time', inplace=True)
    return df




dt = ['20190530', '20210530']
df = ohlcv(dt, 'ETH/BTC', '1m')

In [22]:
print(df)
df.to_csv(filename)

                         Open      High       Low     Close   Volume
Time                                                                
2019-05-30 09:00:00  0.031107  0.031117  0.031097  0.031110   70.714
2019-05-30 09:01:00  0.031111  0.031112  0.031083  0.031101  119.827
2019-05-30 09:02:00  0.031086  0.031101  0.031037  0.031037  272.337
2019-05-30 09:03:00  0.031038  0.031056  0.031030  0.031043  142.499
2019-05-30 09:04:00  0.031045  0.031066  0.031027  0.031065  126.115
...                       ...       ...       ...       ...      ...
2021-05-30 15:57:00  0.067880  0.067899  0.067806  0.067883  112.808
2021-05-30 15:58:00  0.067882  0.067923  0.067873  0.067923   65.316
2021-05-30 15:59:00  0.067920  0.067920  0.067789  0.067789  207.871
2021-05-30 16:00:00  0.067783  0.067817  0.067749  0.067768  102.765
2021-05-30 16:01:00  0.067790  0.067895  0.067786  0.067849  103.700

[2284 rows x 5 columns]


NameError: name 'filename' is not defined

In [31]:
binance = ccxt.gopax()

def min_ohlcv(dt, pair, limit):
    # UTC native object
    since = calendar.timegm(dt.utctimetuple())*1000
    ohlcv1 = binance.fetch_ohlcv(symbol=pair, timeframe='1m', since=since, limit=limit)
    ohlcv2 = binance.fetch_ohlcv(symbol=pair, timeframe='1m', since=since, limit=limit)
    ohlcv = ohlcv1 + ohlcv2
    return ohlcv

def ohlcv(dt, pair, period='1d'):
    ohlcv = []
    limit = 1000
    if period == '1m':
        limit = 720
    elif period == '1d':
        limit = 365
    elif period == '1h':
        limit = 24
    elif period == '5m':
        limit = 288
    for i in dt:
        start_dt = datetime.strptime(i, "%Y%m%d")
        since = calendar.timegm(start_dt.utctimetuple())*1000
        if period == '1m':
            ohlcv.extend(min_ohlcv(start_dt, pair, limit))
        else:
            ohlcv.extend(binance.fetch_ohlcv(symbol=pair, timeframe=period, since=since, limit=limit))
    df = pd.DataFrame(ohlcv, columns = ['Time', 'Open', 'High', 'Low', 'Close', 'Volume'])
    df['Time'] = [datetime.fromtimestamp(float(time)/1000) for time in df['Time']]
    df['Open'] = df['Open'].astype(np.float64)
    df['High'] = df['High'].astype(np.float64)
    df['Low'] = df['Low'].astype(np.float64)
    df['Close'] = df['Close'].astype(np.float64)
    df['Volume'] = df['Volume'].astype(np.float64)
    df.set_index('Time', inplace=True)
    return df




dt = ['2021529', '20210530']
df = ohlcv(dt, 'ETH/KRW', '5m')

In [32]:
print(df)

                          Open       High        Low      Close      Volume
Time                                                                       
2021-05-29 09:00:00  2946000.0  2961000.0  2925000.0  2932000.0  157.048588
2021-05-29 09:05:00  2934000.0  2955000.0  2928000.0  2948000.0   21.188513
2021-05-29 09:10:00  2948000.0  2977000.0  2948000.0  2974000.0    9.146444
2021-05-29 09:15:00  2973000.0  3002000.0  2971000.0  2997000.0   48.980087
2021-05-29 09:20:00  2993000.0  2996000.0  2988000.0  2991000.0   12.207341
...                        ...        ...        ...        ...         ...
2021-05-30 15:40:00  2910000.0  2921000.0  2908000.0  2917000.0   16.051141
2021-05-30 15:45:00  2916000.0  2929000.0  2910000.0  2926000.0   20.410073
2021-05-30 15:50:00  2930000.0  2942000.0  2930000.0  2941000.0   22.297383
2021-05-30 15:55:00  2942000.0  2945000.0  2936000.0  2936000.0   11.055978
2021-05-30 16:00:00  2936000.0  2955000.0  2936000.0  2955000.0   13.646019

[373 rows x

In [ ]:
import ccxt
import pandas as pd

exch = 'binance' # initial exchange
t_frame = '1d' # 1-day timeframe, usually from 1-minute to 1-week depending on the exchange
symbol = 'ADA/BTC' # initial symbol
exchange_list = ['binance','bitfinex','bytetrade','ftx','kraken','poloniex','upbit','acx','bequant','bigone','bitforex','bitkk','bitz','btcalpha','coinex','crex24','digifinex','gateio','hitbtc2','huobipro','huobiru','kucoin','lbank','okex','okex3','stex','upbit','whitebit','zb']
 
# Get our Exchange
try:
    exchange = getattr (ccxt, exch) ()
except AttributeError:
    print('-'*36,' ERROR ','-'*35)
    print('Exchange "{}" not found. Please check the exchange is supported.'.format(exch))
    print('-'*80)
    quit()
 
# Check if fetching of OHLC Data is supported
if exchange.has["fetchOHLCV"] != True:
    print('-'*36,' ERROR ','-'*35)
    print('{} does not support fetching OHLC data. Please use another  exchange'.format(exch))
    print('-'*80)
    quit()
 
# Check requested timeframe is available. If not return a helpful error.
if (not hasattr(exchange, 'timeframes')) or (t_frame not in exchange.timeframes):
    print('-'*36,' ERROR ','-'*35)
    print('The requested timeframe ({}) is not available from {}\n'.format(t_frame,exch))
    print('Available timeframes are:')
    for key in exchange.timeframes.keys():
        print('  - ' + key)
    print('-'*80)
    quit()
 
# Check if the symbol is available on the Exchange
exchange.load_markets()
if symbol not in exchange.symbols:
    print('-'*36,' ERROR ','-'*35)
    print('The requested symbol ({}) is not available from {}\n'.format(symbol,exch))
    print('Available symbols are:')
    for key in exchange.symbols:
        print('  - ' + key)
    print('-'*80)
    quit()
 
 
# Get data
data = exchange.fetch_ohlcv(symbol, t_frame)
header = ['Timestamp', 'Open', 'High', 'Low', 'Close', 'Volume']
df = pd.DataFrame(data, columns=header).set_index('Timestamp')
df['symbol'] = symbol
syms = [symbol]
filename = '{}.csv'.format(t_frame)

for exch in exchange_list:
    try:
        exchange = getattr (ccxt, exch) ()
    except AttributeError:
        print('-'*36,' ERROR ','-'*35)
        print('Exchange "{}" not found. Please check the exchange is supported.'.format(exch))
        print('-'*80)
        quit()
    if exchange.has["fetchOHLCV"] != True:
        print('-'*36,' ERROR ','-'*35)
        print('{} does not support fetching OHLC data. Please use another exchange'.format(exch))
        print('-'*80)
        quit()
    if (not hasattr(exchange, 'timeframes')) or (t_frame not in exchange.timeframes):
        print('-'*36,' ERROR ','-'*35)
        print('The requested timeframe ({}) is not available from {}\n'.format(t_frame,exch))
        print('Available timeframes are:')
        for key in exchange.timeframes.keys():
            print('  - ' + key)
        print('-'*80)
        quit()
    exchange.load_markets()
    for coin in exchange.symbols:
        if coin in syms or coin[-3:] != 'BTC':
            continue
        else:
            try:
                data = exchange.fetch_ohlcv(coin, t_frame)
            except:
                continue
            data_df = pd.DataFrame(data, columns=header).set_index('Timestamp')
            data_df['symbol'] = coin
            df = df.append(data_df)
            syms.append(coin)
df.index = df.index/1000 #Timestamp is 1000 times bigger than it should be in this case
df['Date'] = pd.to_datetime(df.index,unit='s')
df.to_csv(filename)

------------------------------------  ERROR  -----------------------------------
Exchange "acx" not found. Please check the exchange is supported.
--------------------------------------------------------------------------------
